# Quickstart DataFrame

## Create SparkSession

In [1]:
from pyspark.sql import SparkSession, Row, Column
import os
from datetime import datetime
import pandas as pd
from pyspark.sql.functions import upper, pandas_udf

spark = SparkSession.builder.getOrCreate()

25/05/10 20:12:17 WARN Utils: Your hostname, Ubuntu-VMware-Virtual-Platform resolves to a loopback address: 127.0.1.1; using 192.168.48.128 instead (on interface ens33)
25/05/10 20:12:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/10 20:12:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.createDataFrame([
    Row(a=1, b=2., c='string1', d=datetime(2000, 1, 1), e=datetime(2000, 1, 1, 12, 0)),
    Row(a=2, b=3., c='string2', d=datetime(2000, 2, 1), e=datetime(2000, 1, 2, 12, 0)),
    Row(a=4, b=5., c='string3', d=datetime(2000, 3, 1), e=datetime(2000, 1, 3, 12, 0))
])

df

DataFrame[a: bigint, b: double, c: string, d: timestamp, e: timestamp]

In [3]:
# All DataFrames above result same.
df.show()
df.printSchema()

+---+---+-------+-------------------+-------------------+
|  a|  b|      c|                  d|                  e|
+---+---+-------+-------------------+-------------------+
|  1|2.0|string1|2000-01-01 00:00:00|2000-01-01 12:00:00|
|  2|3.0|string2|2000-02-01 00:00:00|2000-01-02 12:00:00|
|  4|5.0|string3|2000-03-01 00:00:00|2000-01-03 12:00:00|
+---+---+-------+-------------------+-------------------+

root
 |-- a: long (nullable = true)
 |-- b: double (nullable = true)
 |-- c: string (nullable = true)
 |-- d: timestamp (nullable = true)
 |-- e: timestamp (nullable = true)



In [4]:
pandas_df = pd.DataFrame({
    'a': [1, 2, 3],
    'b': [2., 3., 4.],
    'c': ['string1', 'string2', 'string3'],
    'd': [datetime(2000, 1, 1), datetime(2000, 2, 1), datetime(2000, 3, 1)],
    'e': [datetime(2000, 1, 1, 12, 0), datetime(2000, 1, 2, 12, 0), datetime(2000, 1, 3, 12, 0)]
})
df = spark.createDataFrame(pandas_df)
df.show()
df.printSchema()

+---+---+-------+-------------------+-------------------+
|  a|  b|      c|                  d|                  e|
+---+---+-------+-------------------+-------------------+
|  1|2.0|string1|2000-01-01 00:00:00|2000-01-01 12:00:00|
|  2|3.0|string2|2000-02-01 00:00:00|2000-01-02 12:00:00|
|  3|4.0|string3|2000-03-01 00:00:00|2000-01-03 12:00:00|
+---+---+-------+-------------------+-------------------+

root
 |-- a: long (nullable = true)
 |-- b: double (nullable = true)
 |-- c: string (nullable = true)
 |-- d: timestamp (nullable = true)
 |-- e: timestamp (nullable = true)



Alternatively, you can enable spark.sql.repl.eagerEval.enabled configuration for the eager evaluation of PySpark DataFrame in notebooks such as Jupyter. The number of rows to show can be controlled via spark.sql.repl.eagerEval.maxNumRows configuration.

In [5]:
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)

In [6]:
df

a,b,c,d,e
1,2.0,string1,2000-01-01 00:00:00,2000-01-01 12:00:00
2,3.0,string2,2000-02-01 00:00:00,2000-01-02 12:00:00
3,4.0,string3,2000-03-01 00:00:00,2000-01-03 12:00:00


In [7]:
df.select("a", "b", "c").describe().show()

+-------+---+---+-------+
|summary|  a|  b|      c|
+-------+---+---+-------+
|  count|  3|  3|      3|
|   mean|2.0|3.0|   NULL|
| stddev|1.0|1.0|   NULL|
|    min|  1|2.0|string1|
|    max|  3|4.0|string3|
+-------+---+---+-------+



## Récuperation des données.

In [8]:
df.collect() # DataFrame.collect() returns a list of Row objects (can be thrown out of memory if too large)

[Row(a=1, b=2.0, c='string1', d=datetime.datetime(2000, 1, 1, 0, 0), e=datetime.datetime(2000, 1, 1, 12, 0)),
 Row(a=2, b=3.0, c='string2', d=datetime.datetime(2000, 2, 1, 0, 0), e=datetime.datetime(2000, 1, 2, 12, 0)),
 Row(a=3, b=4.0, c='string3', d=datetime.datetime(2000, 3, 1, 0, 0), e=datetime.datetime(2000, 1, 3, 12, 0))]

In [9]:
df.take(2) # DataFrame.take(n) returns n Row objects (help to avoid thrown out of memory)

[Row(a=1, b=2.0, c='string1', d=datetime.datetime(2000, 1, 1, 0, 0), e=datetime.datetime(2000, 1, 1, 12, 0)),
 Row(a=2, b=3.0, c='string2', d=datetime.datetime(2000, 2, 1, 0, 0), e=datetime.datetime(2000, 1, 2, 12, 0))]

In [10]:
df.head(2) # DataFrame.head(n) returns first n Row objects

[Row(a=1, b=2.0, c='string1', d=datetime.datetime(2000, 1, 1, 0, 0), e=datetime.datetime(2000, 1, 1, 12, 0)),
 Row(a=2, b=3.0, c='string2', d=datetime.datetime(2000, 2, 1, 0, 0), e=datetime.datetime(2000, 1, 2, 12, 0))]

In [11]:
df.tail(2) # DataFrame.tail(n) returns last n Row objects

[Row(a=2, b=3.0, c='string2', d=datetime.datetime(2000, 2, 1, 0, 0), e=datetime.datetime(2000, 1, 2, 12, 0)),
 Row(a=3, b=4.0, c='string3', d=datetime.datetime(2000, 3, 1, 0, 0), e=datetime.datetime(2000, 1, 3, 12, 0))]

## Columns 

In [12]:
type(df.c) == type(upper(df.c)) == type(df.c.isNull())

True

In [13]:
df.select(df.c).show()

+-------+
|      c|
+-------+
|string1|
|string2|
|string3|
+-------+



In [14]:
# Assign new Column instance.
df.withColumn('upper_c', upper(df.c)).show()

+---+---+-------+-------------------+-------------------+-------+
|  a|  b|      c|                  d|                  e|upper_c|
+---+---+-------+-------------------+-------------------+-------+
|  1|2.0|string1|2000-01-01 00:00:00|2000-01-01 12:00:00|STRING1|
|  2|3.0|string2|2000-02-01 00:00:00|2000-01-02 12:00:00|STRING2|
|  3|4.0|string3|2000-03-01 00:00:00|2000-01-03 12:00:00|STRING3|
+---+---+-------+-------------------+-------------------+-------+



In [15]:
# To select a subset of rows, use DataFrame.filter().
df.filter(df.a > 1).show()

+---+---+-------+-------------------+-------------------+
|  a|  b|      c|                  d|                  e|
+---+---+-------+-------------------+-------------------+
|  2|3.0|string2|2000-02-01 00:00:00|2000-01-02 12:00:00|
|  3|4.0|string3|2000-03-01 00:00:00|2000-01-03 12:00:00|
+---+---+-------+-------------------+-------------------+



In [16]:
df.filter(df.a == 1).show()

+---+---+-------+-------------------+-------------------+
|  a|  b|      c|                  d|                  e|
+---+---+-------+-------------------+-------------------+
|  1|2.0|string1|2000-01-01 00:00:00|2000-01-01 12:00:00|
+---+---+-------+-------------------+-------------------+



## Applying function

In [17]:
@pandas_udf('long')
def pandas_plus_one(series: pd.Series) -> pd.Series:
    # Simply plus one by using pandas Series.
    return series + 1

df.select(pandas_plus_one(df.a)).show()

+------------------+
|pandas_plus_one(a)|
+------------------+
|                 2|
|                 3|
|                 4|
+------------------+



In [18]:
def pandas_filter_func(iterator):
    for pandas_df in iterator:
        yield pandas_df[pandas_df.a == 1]

df.mapInPandas(pandas_filter_func, schema=df.schema).show()

+---+---+-------+-------------------+-------------------+
|  a|  b|      c|                  d|                  e|
+---+---+-------+-------------------+-------------------+
|  1|2.0|string1|2000-01-01 00:00:00|2000-01-01 12:00:00|
+---+---+-------+-------------------+-------------------+



## Grouping

In [19]:
df = spark.createDataFrame([
    ['red', 'banana', 1, 10], ['blue', 'banana', 2, 20], ['red', 'carrot', 3, 30],
    ['blue', 'grape', 4, 40], ['red', 'carrot', 5, 50], ['black', 'carrot', 6, 60],
    ['red', 'banana', 7, 70], ['red', 'grape', 8, 80]], schema=['color', 'fruit', 'v1', 'v2'])
df.show()

+-----+------+---+---+
|color| fruit| v1| v2|
+-----+------+---+---+
|  red|banana|  1| 10|
| blue|banana|  2| 20|
|  red|carrot|  3| 30|
| blue| grape|  4| 40|
|  red|carrot|  5| 50|
|black|carrot|  6| 60|
|  red|banana|  7| 70|
|  red| grape|  8| 80|
+-----+------+---+---+



In [20]:
df.groupBy('color').avg().show()

+-----+-------+-------+
|color|avg(v1)|avg(v2)|
+-----+-------+-------+
|  red|    4.8|   48.0|
| blue|    3.0|   30.0|
|black|    6.0|   60.0|
+-----+-------+-------+



In [21]:
df.groupBy('color').avg('v1', 'v2').show()

+-----+-------+-------+
|color|avg(v1)|avg(v2)|
+-----+-------+-------+
|  red|    4.8|   48.0|
| blue|    3.0|   30.0|
|black|    6.0|   60.0|
+-----+-------+-------+



In [22]:
df.groupBy('color').agg({'v1': 'avg', 'v2': 'sum'}).show()

+-----+-------+-------+
|color|avg(v1)|sum(v2)|
+-----+-------+-------+
|  red|    4.8|    240|
| blue|    3.0|     60|
|black|    6.0|     60|
+-----+-------+-------+



In [23]:
def plus_mean(pandas_df):
    return pandas_df.assign(v1=pandas_df.v1 - pandas_df.v1.mean())

df.groupby('color').applyInPandas(plus_mean, schema=df.schema).show()

+-----+------+---+---+
|color| fruit| v1| v2|
+-----+------+---+---+
|black|carrot|  0| 60|
| blue|banana| -1| 20|
| blue| grape|  1| 40|
|  red|banana| -3| 10|
|  red|carrot| -1| 30|
|  red|carrot|  0| 50|
|  red|banana|  2| 70|
|  red| grape|  3| 80|
+-----+------+---+---+



In [25]:
df1 = spark.createDataFrame(
    [(20000101, 1, 1.0), (20000101, 2, 2.0), (20000102, 1, 3.0), (20000102, 2, 4.0)],
    ('time', 'id', 'v1'))

df2 = spark.createDataFrame(
    [(20000101, 1, 'x'), (20000101, 2, 'y')],
    ('time', 'id', 'v2'))

df1.show()
df2.show()

def merge_ordered(l, r):
    return pd.merge_ordered(l, r)

df1.groupby('id').cogroup(df2.groupby('id')).applyInPandas(merge_ordered, schema='time int, id int, v1 double, v2 string').show()

+--------+---+---+
|    time| id| v1|
+--------+---+---+
|20000101|  1|1.0|
|20000101|  2|2.0|
|20000102|  1|3.0|
|20000102|  2|4.0|
+--------+---+---+

+--------+---+---+
|    time| id| v2|
+--------+---+---+
|20000101|  1|  x|
|20000101|  2|  y|
+--------+---+---+



+--------+---+---+----+
|    time| id| v1|  v2|
+--------+---+---+----+
|20000101|  1|1.0|   x|
|20000102|  1|3.0|NULL|
|20000101|  2|2.0|   y|
|20000102|  2|4.0|NULL|
+--------+---+---+----+



## Getting Data In/Out

In [26]:
df.write.csv('foo.csv', header=True)
spark.read.csv('foo.csv', header=True).show()

+-----+------+---+---+
|color| fruit| v1| v2|
+-----+------+---+---+
|black|carrot|  6| 60|
| blue|banana|  2| 20|
|  red|carrot|  3| 30|
|  red|banana|  7| 70|
| blue| grape|  4| 40|
|  red|banana|  1| 10|
|  red|carrot|  5| 50|
|  red| grape|  8| 80|
+-----+------+---+---+



In [28]:
df.write.parquet('bar.parquet')
spark.read.parquet('bar.parquet').show()

25/05/10 20:34:32 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


+-----+------+---+---+
|color| fruit| v1| v2|
+-----+------+---+---+
|black|carrot|  6| 60|
| blue|banana|  2| 20|
|  red|banana|  7| 70|
|  red|carrot|  5| 50|
| blue| grape|  4| 40|
|  red|carrot|  3| 30|
|  red|banana|  1| 10|
|  red| grape|  8| 80|
+-----+------+---+---+



In [29]:
df.write.orc('zoo.orc')
spark.read.orc('zoo.orc').show()

+-----+------+---+---+
|color| fruit| v1| v2|
+-----+------+---+---+
|  red|banana|  7| 70|
|  red| grape|  8| 80|
|black|carrot|  6| 60|
| blue|banana|  2| 20|
|  red|banana|  1| 10|
|  red|carrot|  5| 50|
|  red|carrot|  3| 30|
| blue| grape|  4| 40|
+-----+------+---+---+



## Working with SQL
DataFrame and Spark SQL share the same execution engine so they can be interchangeably used seamlessly. For example, you can register the DataFrame as a table and run a SQL easily as below

In [30]:
df.createOrReplaceTempView("tableA")
spark.sql("SELECT count(*) from tableA").show()

+--------+
|count(1)|
+--------+
|       8|
+--------+



In [31]:
spark.sql("select * from tableA;").show()

+-----+------+---+---+
|color| fruit| v1| v2|
+-----+------+---+---+
|  red|banana|  1| 10|
| blue|banana|  2| 20|
|  red|carrot|  3| 30|
| blue| grape|  4| 40|
|  red|carrot|  5| 50|
|black|carrot|  6| 60|
|  red|banana|  7| 70|
|  red| grape|  8| 80|
+-----+------+---+---+



In [34]:
@pandas_udf("integer")
def multi_two_col(col1: pd.Series, col2: pd.Series) -> pd.Series:
    return col1 * col2

spark.udf.register("multi_two_col", multi_two_col)
spark.sql("select multi_two_col(v1, v2) as v3 from tableA").show()

25/05/10 20:47:47 WARN SimpleFunctionRegistry: The function multi_two_col replaced a previously registered function.


+---+
| v3|
+---+
| 10|
| 40|
| 90|
|160|
|250|
|360|
|490|
|640|
+---+



In [35]:
@pandas_udf("integer")
def add_one(s: pd.Series) -> pd.Series:
    return s + 1

spark.udf.register("add_one", add_one)
spark.sql("SELECT add_one(v1) FROM tableA").show()

+-----------+
|add_one(v1)|
+-----------+
|          2|
|          3|
|          4|
|          5|
|          6|
|          7|
|          8|
|          9|
+-----------+



These SQL expressions can directly be mixed and used as PySpark columns.

In [36]:
from pyspark.sql.functions import expr

df.selectExpr('add_one(v1)').show()
df.select(expr('count(*)') > 0).show()

+-----------+
|add_one(v1)|
+-----------+
|          2|
|          3|
|          4|
|          5|
|          6|
|          7|
|          8|
|          9|
+-----------+



+--------------+
|(count(1) > 0)|
+--------------+
|          true|
+--------------+

